# Who Is Missing: Bode, Data Feminism, and the Absence Audit

**Computational Text Analysis Pack · Module 22 — Distant Reading and Its Critics**

*Technique:* Corpus critique as method; auditing what a corpus cannot say

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/11-distant-reading-and-its-critics/22c_who_is_missing.ipynb)

Data: `colonial_texts.csv`, `masakhanews.csv`, `african_legal.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv', 'masakhanews.csv', 'african_legal.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
df       = pd.read_csv('masakhanews.csv')
df_legal = pd.read_csv('african_legal.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

Notebook 11c tested whether our findings were statistically robust. This one
asks a question no significance test reaches: **robust about whom?**

Three critiques, all live:

- **Katherine Bode (2017)** argues Moretti and Jockers model literary systems
  *ahistorically*, treating digitised text as though it were the literary field
  rather than a heavily mediated survival of it. Her alternative is the scholarly
  edition as the proper data-rich object.
- **D'Ignazio and Klein, *Data Feminism*** — ask who collected the data, who is
  counted, and who is not.
- **The "great unread"** — the vast majority of published writing was never
  digitised, and what survives is not a random sample.

The method is an **absence audit**, and unlike the critiques themselves it is
something you can run.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
print("=== WHAT THE COLONIAL CORPUS IS ===\n")
print(f"passages : {len(df_colonial):,}")
print(f"works    : {df_colonial['work_id'].nunique()}")
print(f"authors  : {df_colonial['author'].nunique()}")
print()
print("Top authors by share of the corpus:")
share = df_colonial.groupby("author").size().sort_values(ascending=False)
for a, n in share.head(10).items():
    bar = "#" * int(40 * n / share.max())
    print(f"  {str(a)[:40]:<42}{n:>5}  {bar}")
print()
print(f"Gini-style concentration: the top 10% of authors hold "
      f"{share.head(max(1, len(share)//10)).sum()/len(df_colonial):.0%} of passages")

## The audit's first question: who wrote this?

This corpus came from a Project Gutenberg keyword search for "africa". The
`PROVENANCE.md` shipped with it is blunt about the consequence, and it is worth
restating because it is the single most important fact about the dataset:

> The colonial-era corpus is a **colonial-discourse** collection, not an
> African-authored one. Any claim from it is about how Africa was *written
> about*, in English, before 1929.

That sentence is an absence audit in one line. Everything below makes it
concrete.

In [ ]:
# Which names appear as authors, and what that tells us.
authors = sorted(df_colonial["author"].dropna().unique())
print(f"{len(authors)} distinct author strings. A sample:\n")
for a in authors[:18]:
    print(f"   {a}")
print()
print("Read that list as evidence. It is overwhelmingly British and American,")
print("nineteenth-century, and institutional ('Work Projects Administration').")
print("The people the texts are ABOUT are not among the people who WROTE them,")
print("and no amount of careful statistics on this corpus will fix that.")

In [ ]:
# Non-authorial entries: the corpus contains categories that are not people.
suspicious = [a for a in authors if any(k in str(a).lower() for k in
              ["various", "unknown", "administration", "society", "company",
               "office", "committee", "anonymous"])]
n_susp = df_colonial[df_colonial["author"].isin(suspicious)].shape[0]
print(f"{len(suspicious)} 'author' values are institutions or placeholders, "
      f"covering {n_susp:,} passages ({n_susp/len(df_colonial):.1%})\n")
for s in suspicious[:10]:
    print(f"   {s}")
print()
print("Any per-author analysis — including this course's own stylometry module —")
print("silently treats these as people. Module 9 filtered them out by requiring")
print("two works per author, which worked by luck rather than by design.")

## Comparing what three corpora can and cannot say

The pack ships three datasets with very different politics of collection. Laying
their properties side by side is the clearest way to see that a research question
must be matched to a corpus, not the other way round.

In [ ]:
summary = []
for name, frame, who in [
    ("colonial_texts", df_colonial, "European/American authors writing about Africa"),
    ("masakhanews", df, "African journalists, 16 languages"),
    ("african_legal", df_legal, "institutions: ECOWAS, ANC, TRC"),
]:
    col = "text"
    toks = sum(len(str(t).split()) for t in frame[col].head(500))
    summary.append({"corpus": name, "rows": len(frame),
                    "words_per_row": toks // min(500, len(frame)),
                    "voice": who})
print(pd.DataFrame(summary).to_string(index=False))
print()
print("A question about African literary style cannot be answered by the first;")
print("a question about pre-1929 discourse cannot be answered by the second; a")
print("question about ordinary speech cannot be answered by any of them.")

In [ ]:
# The absence audit as a repeatable procedure.
langs = df["language"].value_counts()
plt.figure(figsize=(8, 4))
plt.barh(langs.index[::-1], langs.values[::-1], color="steelblue")
plt.xlabel("documents")
plt.title("MasakhaNEWS: which languages are represented, and how unequally")
plt.tight_layout()

print("Languages in the news corpus:", len(langs))
print(f"largest / smallest ratio: {langs.max() / langs.min():.1f}x\n")
print("Africa has over 2,000 languages. This corpus has", len(langs), "-")
print("and the ones present are those with a digitised news industry, which is")
print("itself a consequence of colonial language policy. The corpus inherits")
print("that history whether or not the analysis mentions it.")

## The audit, as a checklist you can reuse

Run these five questions against any corpus before you publish a number from it.
They are the operational form of Bode's and Data Feminism's critiques.

1. **Who produced these texts, and who did not?** Name the excluded group.
2. **How did the texts survive to be digitised?** Selection happened at least
   twice: what was printed, and what was scanned.
3. **What does one row represent?** A book, a passage, an article — and who
   chose the boundaries.
4. **Which categories are people, and which are institutions?** Check before any
   per-author analysis.
5. **What claim would this corpus refute?** If none, the corpus cannot support
   the opposite claim either.

The honest answer to (5) is often "none", and finding that out before the
analysis is cheaper than finding it out in peer review.

### 🔧 Your turn 1

Run all five questions against the corpus you plan to use for the capstone. Write the answers as a paragraph, and keep it — it is the methods section you will need anyway.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

The paragraph is usually harder to write than the analysis, and that difficulty
is the finding. Bode's argument is that this work — establishing what the data
*is*, historically and bibliographically — is not preliminary to computational
literary study but constitutive of it, and skipping it is what makes results
look more general than they are. Note also that this is where a course on
African DH departs from the standard curriculum: for most of these corpora the
answer to question 1 involves a colonial archive, and stating that plainly is
better scholarship than a hedge in a footnote.

</details>